In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

# Participants de la cohorte (359).
participants = pd.read_csv(C.PARTICIPANTS_359, dtype={"PTID": "string"})
ptids = set(participants["PTID"].str.strip().str.upper())
print("Cohorte :", len(ptids), "participants")

# --- Historique diagnostique (une ligne = une visite) ---
dx = pd.read_csv(C.table("All_Subjects_DXSUM_"), dtype="string", encoding="utf-8-sig")
dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(dx["EXAMDATE"].str.strip(),
                               format="%Y-%m-%d", errors="coerce")
dx = dx.loc[
    dx["PTID"].isin(ptids) & dx["DIAGNOSIS"].isin(["1", "2", "3"])
    & dx["date_dx"].notna(), ["PTID", "date_dx", "DIAGNOSIS"]]

# Ecarter les visites a codes contradictoires.
conf = dx.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
conf = conf[conf > 1].index
dx = dx.loc[~pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]]).isin(conf)]
dx = dx.drop_duplicates(subset=["PTID", "date_dx"]).copy()
dx["code"] = dx["DIAGNOSIS"].astype(int)

visites = dx[["PTID", "date_dx", "code"]].sort_values(["PTID", "date_dx"])
print("Visites diagnostiques :", len(visites))

In [ ]:
# --- Attacher les scores cognitifs a chaque visite (+/- 90 jours) ---
def attacher(base, prefixe, col_date, colonnes):
    t = pd.read_csv(C.table(prefixe), dtype="string", encoding="utf-8-sig")
    t["PTID"] = t["PTID"].str.strip().str.upper()
    t["date"] = pd.to_datetime(t[col_date].str.strip(), errors="coerce").dt.normalize()
    noms = []
    for brut, nom in colonnes.items():
        t[nom] = pd.to_numeric(t[brut], errors="coerce")
        noms.append(nom)
    t = (t.loc[t["PTID"].isin(ptids)]
           .dropna(subset=["PTID", "date"])[["PTID", "date"] + noms]
           .dropna(subset=noms, how="all")
           .drop_duplicates(subset=["PTID", "date"])
           .sort_values("date"))
    lien = pd.merge_asof(
        base.sort_values("date_dx"), t,
        by="PTID", left_on="date_dx", right_on="date",
        direction="nearest", tolerance=pd.Timedelta(days=90))
    return lien[["PTID", "date_dx"] + noms]

panel = visites.copy()
for prefixe, cd, cols in [
    ("All_Subjects_MMSE_", "VISDATE", {"MMSCORE": "MMSE"}),
    ("All_Subjects_CDR_",  "VISDATE", {"CDGLOBAL": "CDR", "CDRSB": "CDR_SB"}),
    ("All_Subjects_ADAS_", "VISDATE", {"TOTAL13": "ADAS13"}),
    ("All_Subjects_FAQ_",  "VISDATE", {"FAQTOTAL": "FAQ"}),
]:
    ajout = attacher(visites, prefixe, cd, cols)
    panel = panel.merge(ajout, on=["PTID", "date_dx"], how="left")

VARIABLES = ["code", "MMSE", "CDR", "CDR_SB", "ADAS13", "FAQ"]
panel = panel.sort_values(["PTID", "date_dx"]).reset_index(drop=True)
print("Panel (visites x variables) :", panel.shape)
print("Taux de remplissage par variable :")
print((panel[VARIABLES].notna().mean() * 100).round(1).to_string())

In [ ]:
# --- Construire les sequences : 3 visites d'entree, cible = aggravation apres ---
K = 3  # nombre de visites utilisees en entree

panel["rang"] = panel.groupby("PTID").cumcount()
nb_visites = panel.groupby("PTID")["rang"].max() + 1

# Il faut au moins K visites d'entree + 1 visite de suivi.
eligibles = sorted(nb_visites[nb_visites >= K + 1].index)
print("Participants avec au moins", K + 1, "visites :", len(eligibles))

p = panel.loc[panel["PTID"].isin(eligibles)]

code_depart = p.loc[p["rang"] == K - 1].set_index("PTID")["code"]
code_futur = p.loc[p["rang"] >= K].groupby("PTID")["code"].max()

cible = (code_futur.reindex(eligibles) > code_depart.reindex(eligibles)).astype(int)

# Tableau des sequences : (n_participants, K visites, n_variables)
entree = p.loc[p["rang"] < K].sort_values(["PTID", "rang"])
X_seq = (entree.set_index(["PTID", "rang"])[VARIABLES]
         .reindex(pd.MultiIndex.from_product([eligibles, range(K)],
                                             names=["PTID", "rang"]))
         .to_numpy(dtype=float)
         .reshape(len(eligibles), K, len(VARIABLES)))

y = cible.to_numpy()
print("Forme des sequences :", X_seq.shape)
print("Participants :", len(y), "| aggravation apres la visite", K, ":",
      int(y.sum()), f"({y.mean() * 100:.1f}%)")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
NV = len(VARIABLES)

def preparer(X_app, X_test):
    """Imputation (mediane d'apprentissage) puis standardisation."""
    plat_app = X_app.reshape(-1, NV)
    mediane = np.nanmedian(plat_app, axis=0)
    mediane = np.where(np.isnan(mediane), 0.0, mediane)

    def remplir(X):
        X = X.copy()
        for j in range(NV):
            colonne = X[:, :, j]
            colonne[np.isnan(colonne)] = mediane[j]
            X[:, :, j] = colonne
        return X

    A, T = remplir(X_app), remplir(X_test)
    moy = A.reshape(-1, NV).mean(axis=0)
    ect = A.reshape(-1, NV).std(axis=0) + 1e-6
    return (A - moy) / ect, (T - moy) / ect

def aplatir_avec_pentes(X):
    """Les K visites mises bout a bout + la pente de chaque variable."""
    pentes = X[:, -1, :] - X[:, 0, :]
    return np.concatenate([X.reshape(len(X), -1), pentes], axis=1)

In [ ]:
class BiLSTM(nn.Module):
    """Petit BiLSTM : lit la sequence des visites et predit l'aggravation."""
    def __init__(self, n_variables, cache=16):
        super().__init__()
        self.lstm = nn.LSTM(n_variables, cache, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.sortie = nn.Linear(cache * 2, 1)

    def forward(self, x):
        suite, _ = self.lstm(x)
        return self.sortie(self.drop(suite[:, -1, :])).squeeze(1)

def entrainer_bilstm(X_app, y_app, X_test, epoques=150):
    torch.manual_seed(0)
    modele = BiLSTM(NV)
    poids = torch.tensor([(len(y_app) - y_app.sum()) / max(y_app.sum(), 1)],
                         dtype=torch.float32)
    perte = nn.BCEWithLogitsLoss(pos_weight=poids)
    optim = torch.optim.Adam(modele.parameters(), lr=0.01, weight_decay=1e-4)

    xa = torch.tensor(X_app, dtype=torch.float32)
    ya = torch.tensor(y_app, dtype=torch.float32)

    modele.train()
    for _ in range(epoques):
        optim.zero_grad()
        sortie = perte(modele(xa), ya)
        sortie.backward()
        optim.step()

    modele.eval()
    with torch.no_grad():
        proba = torch.sigmoid(modele(torch.tensor(X_test, dtype=torch.float32)))
    return modele, proba.numpy()

In [ ]:
# --- Validation croisee : tous les modeles sur les MEMES plis ---
resultats = {nom: [] for nom in [
    "Statique — random forest",
    "Statique — regression logistique",
    "Longitudinal — random forest (3 visites + pentes)",
    "Longitudinal — BiLSTM (deep learning)",
]}
rq3 = {"BiLSTM complet": [], "BiLSTM sans la visite 2": []}

for app, test in CV.split(X_seq, y):
    Xa, Xt = preparer(X_seq[app], X_seq[test])
    ya, yt = y[app], y[test]

    # 1-2. Statique : uniquement la derniere visite d'entree.
    Sa, St = Xa[:, -1, :], Xt[:, -1, :]
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=0).fit(Sa, ya)
    resultats["Statique — random forest"].append(
        roc_auc_score(yt, rf.predict_proba(St)[:, 1]))

    lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Sa, ya)
    resultats["Statique — regression logistique"].append(
        roc_auc_score(yt, lr.predict_proba(St)[:, 1]))

    # 3. Longitudinal classique.
    Fa, Ft = aplatir_avec_pentes(Xa), aplatir_avec_pentes(Xt)
    rfl = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                 random_state=0).fit(Fa, ya)
    resultats["Longitudinal — random forest (3 visites + pentes)"].append(
        roc_auc_score(yt, rfl.predict_proba(Ft)[:, 1]))

    # 4. BiLSTM.
    modele, proba = entrainer_bilstm(Xa, ya, Xt)
    auc_lstm = roc_auc_score(yt, proba)
    resultats["Longitudinal — BiLSTM (deep learning)"].append(auc_lstm)

    # RQ3 : masquer la visite du milieu au test (valeur neutre = 0 apres
    # standardisation, c'est-a-dire la moyenne d'apprentissage).
    Xt_masque = Xt.copy()
    Xt_masque[:, 1, :] = 0.0
    with torch.no_grad():
        p_masque = torch.sigmoid(
            modele(torch.tensor(Xt_masque, dtype=torch.float32))).numpy()
    rq3["BiLSTM complet"].append(auc_lstm)
    rq3["BiLSTM sans la visite 2"].append(roc_auc_score(yt, p_masque))

print("Validation croisee terminee.")

In [ ]:
# --- RQ2 : tableau comparatif statique vs longitudinal ---
tableau = pd.DataFrame({
    "ROC-AUC": {nom: f"{np.mean(v):.3f} ± {np.std(v):.3f}"
                for nom, v in resultats.items()},
    "_moyenne": {nom: np.mean(v) for nom, v in resultats.items()},
}).sort_values("_moyenne", ascending=False)
moyennes = tableau["_moyenne"]
tableau = tableau[["ROC-AUC"]]
tableau.index.name = "Modele"

print("=== RQ2 — Statique vs longitudinal ===")
print("Participants :", len(y), "| visites d'entree :", K)
display(tableau)

meilleur_statique = max(m for n, m in moyennes.items() if n.startswith("Statique"))
meilleur_longi = max(m for n, m in moyennes.items() if n.startswith("Longitudinal"))
print(f"Meilleur statique     : AUC {meilleur_statique:.3f}")
print(f"Meilleur longitudinal : AUC {meilleur_longi:.3f}")
print("Apport du longitudinal :", round(meilleur_longi - meilleur_statique, 3), "d'AUC")

In [ ]:
# --- RQ3 : robustesse a une visite manquante ---
robustesse = pd.DataFrame({
    "situation": list(rq3),
    "ROC-AUC": [f"{np.mean(v):.3f} ± {np.std(v):.3f}" for v in rq3.values()],
    "_m": [np.mean(v) for v in rq3.values()],
})
perte = robustesse["_m"].iloc[0] - robustesse["_m"].iloc[1]
robustesse = robustesse[["situation", "ROC-AUC"]]

print("=== RQ3 — Une visite manquante ===")
display(robustesse)
print("Perte d'AUC quand une visite manque :", round(perte, 3))

# --- Enregistrement ---
SORTIE = C.sortie("longitudinal_RQ2_RQ3")
tableau.to_csv(SORTIE / "RQ2_statique_vs_longitudinal.csv", encoding="utf-8-sig")
robustesse.to_csv(SORTIE / "RQ3_visite_manquante.csv", index=False, encoding="utf-8-sig")
print("\nResultats enregistres dans :", SORTIE)

In [ ]:
import matplotlib.pyplot as plt

# Reutiliser le dossier de sortie du notebook 12.
SORTIE = sorted(C.COHORTE.glob("longitudinal_RQ2_RQ3_*"))[-1]

# 1) Le longitudinal gagne-t-il dans chaque pli ?
par_pli = pd.DataFrame(resultats)
par_pli.index = [f"pli {i + 1}" for i in range(len(par_pli))]

cols_longi = [c for c in par_pli.columns if c.startswith("Longitudinal")]
cols_stat = [c for c in par_pli.columns if c.startswith("Statique")]

par_pli["meilleur longitudinal"] = par_pli[cols_longi].max(axis=1)
par_pli["meilleur statique"] = par_pli[cols_stat].max(axis=1)
par_pli["ecart"] = par_pli["meilleur longitudinal"] - par_pli["meilleur statique"]
par_pli["longitudinal gagne"] = par_pli["ecart"] > 0

display(par_pli.round(3))

victoires = int(par_pli["longitudinal gagne"].sum())
print(f"Le longitudinal gagne dans {victoires} plis sur {len(par_pli)}")
print(f"Ecart apparie moyen : {par_pli['ecart'].mean():+.3f} "
      f"± {par_pli['ecart'].std():.3f} d'AUC")
print("(5 plis : trop peu pour un test statistique formel ;")
print(" on rapporte la coherence du sens de l'ecart.)")

# 2) Graphique RQ2 + RQ3
noms = sorted(resultats, key=lambda n: np.mean(resultats[n]), reverse=True)
moy = [np.mean(resultats[n]) for n in noms]
ect = [np.std(resultats[n]) for n in noms]
couleurs = ["#4C72B0" if n.startswith("Longitudinal") else "#B0B0B0" for n in noms]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True,
                               gridspec_kw={"width_ratios": [2.2, 1]})
ypos = np.arange(len(noms))[::-1]
ax1.barh(ypos, moy, xerr=ect, color=couleurs, capsize=4)
ax1.set_yticks(ypos)
ax1.set_yticklabels([n.replace(" — ", "\n") for n in noms], fontsize=8)
ax1.set_xlim(0.45, 1.0)
ax1.axvline(0.5, color="grey", ls="--", lw=1)
for yp, m, e in zip(ypos, moy, ect):
    ax1.text(m + e + 0.015, yp, f"{m:.3f}", va="center", fontsize=9)
ax1.set_xlabel("ROC-AUC (validation croisee 5 plis)")
ax1.set_title("RQ2 — statique (gris) vs longitudinal (bleu)")

n3 = list(rq3)
m3 = [np.mean(rq3[n]) for n in n3]
e3 = [np.std(rq3[n]) for n in n3]
ax2.bar(range(len(n3)), m3, yerr=e3, color=["#4C72B0", "#DD8452"], capsize=4)
ax2.set_xticks(range(len(n3)))
ax2.set_xticklabels(["complet", "sans\nvisite 2"], fontsize=9)
ax2.set_ylim(0.45, 1.0)
ax2.axhline(0.5, color="grey", ls="--", lw=1)
for i, (m, e) in enumerate(zip(m3, e3)):
    ax2.text(i, m + e + 0.02, f"{m:.3f}", ha="center", fontsize=9)
ax2.set_ylabel("ROC-AUC")
ax2.set_title("RQ3 — visite manquante")

fig.savefig(SORTIE / "RQ2_RQ3_graphique.png", dpi=150)
plt.show()

par_pli.round(4).to_csv(SORTIE / "RQ2_par_pli.csv", encoding="utf-8-sig")
print("\nGraphique et tableau par pli enregistres dans :", SORTIE.name)